In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from google.colab import files
existing=list(Path("/content").glob("Day15_Executive_Hotel_Booking_EDA_Dataset.csv"))
if existing:
    file_path=str(existing[0])
else:
    uploaded=files.upload()
    file_path=next(iter(uploaded))
raw=pd.read_csv(file_path)
raw.head()

In [ ]:
print("Original shape:", raw.shape)
display(raw.dtypes.to_frame("Data Type"))
display(raw.isna().sum().sort_values(ascending=False).to_frame("Missing Values"))
print("Duplicate rows:", raw.duplicated().sum())
display(raw.describe(include="all").T)

In [ ]:
df=raw.copy()
for c in ["Booking_Date","Arrival_Date","Reservation_Status_Date"]:
    df[c]=pd.to_datetime(df[c],errors="coerce")

df=df.drop_duplicates().copy()

def norm_text(x):
    if pd.isna(x): return np.nan
    return str(x).strip()

for c in ["Hotel_Type","Hotel_Location","Market_Segment","Distribution_Channel",
          "Deposit_Type","Customer_Type","Meal_Type","Reservation_Status","Country"]:
    df[c]=df[c].map(norm_text)

df["Hotel_Type"]=df["Hotel_Type"].str.lower().replace({"city hotel":"City Hotel","resort hotel":"Resort Hotel"})
df["Market_Segment"]=df["Market_Segment"].str.lower().replace({
    "online ta":"Online TA","offline ta/to":"Offline TA/TO","direct":"Direct","groups":"Groups"
})
df["Distribution_Channel"]=df["Distribution_Channel"].str.lower().replace({
    "ta/to":"TA/TO","direct":"Direct","corporate":"Corporate"
})

for c in ["Children","Agent_ID","Company_ID","Satisfaction_Score","ADR"]:
    df[c]=df[c].fillna(df[c].median())
for c in ["Country","Hotel_Location","Meal_Type"]:
    df[c]=df[c].fillna(df[c].mode()[0])

nonnegative_cols=["Lead_Time_Days","Weekend_Nights","Weekday_Nights","Adults","Children","Babies",
                  "Previous_Cancellations","Previous_Bookings","Booking_Changes","Days_In_Waiting_List",
                  "Total_Nights","ADR","Required_Car_Parking_Spaces","Total_Special_Requests","Estimated_Revenue"]
for c in nonnegative_cols:
    df=df[df[c]>=0]

for c in ["Lead_Time_Days","ADR","Estimated_Revenue","Total_Nights"]:
    q1,q3=df[c].quantile([.25,.75]); iqr=q3-q1
    lo=q1-1.5*iqr; hi=q3+1.5*iqr
    df[c]=df[c].clip(lo,hi)

df["Total_Guests"]=df["Adults"]+df["Children"]+df["Babies"]
df["Stay_Type"]=np.where(df["Total_Nights"]==0,"No Stay",np.where(df["Total_Nights"]==1,"1 Night","2+ Nights"))
df["Arrival_Year"]=df["Arrival_Date"].dt.year
df["Arrival_Month"]=df["Arrival_Date"].dt.month

print("Cleaned shape:",df.shape)
print("Remaining missing values:",int(df.isna().sum().sum()))
print("Remaining duplicates:",df.duplicated().sum())

In [ ]:
display(df.describe().T)
display(df.select_dtypes(exclude=np.number).nunique().to_frame("Unique Values"))

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
sns.countplot(data=df,x="Hotel_Type",hue="Is_Canceled",ax=ax)
ax.set_title("Cancellation Count by Hotel Type")
ax.set_xlabel("Hotel Type"); ax.set_ylabel("Bookings")
ax.legend(title="Canceled",labels=["No","Yes"])
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sns.histplot(data=df,x="Lead_Time_Days",bins=30,kde=True,ax=ax)
ax.set_title("Distribution of Lead Time")
ax.set_xlabel("Lead Time (Days)"); ax.set_ylabel("Frequency")
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sns.boxplot(data=df,x="Hotel_Type",y="ADR",ax=ax)
ax.set_title("ADR Distribution by Hotel Type")
ax.set_xlabel("Hotel Type"); ax.set_ylabel("Average Daily Rate")
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sns.violinplot(data=df,x="Hotel_Type",y="Satisfaction_Score",inner="quartile",ax=ax)
ax.set_title("Satisfaction Score Distribution by Hotel Type")
ax.set_xlabel("Hotel Type"); ax.set_ylabel("Satisfaction Score")
plt.tight_layout(); plt.show()

In [ ]:
segment_summary=df.groupby("Market_Segment").agg(
    Bookings=("Booking_ID","count"),
    Cancel_Rate=("Is_Canceled","mean"),
    Avg_Revenue=("Estimated_Revenue","mean")
).sort_values("Cancel_Rate",ascending=False)
display(segment_summary.round(3))

fig,ax=plt.subplots(figsize=(10,5))
sns.barplot(data=segment_summary.reset_index(),x="Market_Segment",y="Cancel_Rate",ax=ax)
ax.set_title("Cancellation Rate by Market Segment")
ax.set_xlabel("Market Segment"); ax.set_ylabel("Cancellation Rate")
ax.tick_params(axis="x",rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
hotel_summary=df.groupby("Hotel_Type").agg(
    Bookings=("Booking_ID","count"),
    Cancel_Rate=("Is_Canceled","mean"),
    Avg_ADR=("ADR","mean"),
    Avg_Revenue=("Estimated_Revenue","mean"),
    Avg_Lead_Time=("Lead_Time_Days","mean"),
    Avg_Satisfaction=("Satisfaction_Score","mean")
).sort_values("Avg_Revenue",ascending=False)
display(hotel_summary.round(3))

fig,ax=plt.subplots(figsize=(8,5))
sns.barplot(data=hotel_summary.reset_index(),x="Hotel_Type",y="Avg_Revenue",ax=ax)
ax.set_title("Average Estimated Revenue by Hotel Type")
ax.set_xlabel("Hotel Type"); ax.set_ylabel("Average Estimated Revenue")
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
sns.scatterplot(data=df,x="Lead_Time_Days",y="Is_Canceled",alpha=.25,ax=ax)
sns.regplot(data=df,x="Lead_Time_Days",y="Is_Canceled",scatter=False,ax=ax)
ax.set_title("Lead Time vs Cancellation")
ax.set_xlabel("Lead Time (Days)"); ax.set_ylabel("Cancellation (0=No, 1=Yes)")
plt.tight_layout(); plt.show()
print(f"Pearson correlation: {df['Lead_Time_Days'].corr(df['Is_Canceled']):.3f}")

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
sns.scatterplot(data=df,x="ADR",y="Estimated_Revenue",hue="Hotel_Type",alpha=.6,ax=ax)
ax.set_title("ADR vs Estimated Revenue")
ax.set_xlabel("ADR"); ax.set_ylabel("Estimated Revenue")
ax.legend(title="Hotel Type")
plt.tight_layout(); plt.show()

In [ ]:
corr_cols=["Lead_Time_Days","Weekend_Nights","Weekday_Nights","Adults","Children",
           "Total_Nights","ADR","Total_Special_Requests","Satisfaction_Score",
           "Estimated_Revenue","Is_Canceled"]
corr=df[corr_cols].corr()
plt.figure(figsize=(11,8))
sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm",center=0)
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.show()
display(corr.round(3))

In [ ]:
print("EXECUTIVE INSIGHTS")
print("1. The cleaned dataset contains",len(df),"booking records; the overall cancellation rate is",f"{df['Is_Canceled'].mean()*100:.1f}%." )
print("2. Lead time has a correlation of",f"{df['Lead_Time_Days'].corr(df['Is_Canceled']):.3f}","with cancellation, making booking horizon a major cancellation-risk indicator.")
print("3. Resort Hotel has substantially higher average ADR and estimated revenue than City Hotel.")
print("4. Online TA has the highest cancellation rate among the major market segments in the cleaned data.")
print("5. Revenue is strongly linked to ADR and length/stay-related volume measures, making pricing and occupancy drivers central to commercial performance.")

In [ ]:
print("MANAGEMENT RECOMMENDATIONS")
recommendations=[
"1. Introduce stronger cancellation-risk controls for long-lead bookings, such as flexible deposits with non-refundable or partially refundable options.",
"2. Use segment-specific cancellation policies, with particular attention to Online Travel Agency bookings.",
"3. Protect premium pricing at Resort Hotels while testing targeted upselling and package offers to increase revenue per stay.",
"4. Use ADR, length of stay, and booking lead time together for revenue-management decisions rather than relying on ADR alone.",
"5. Monitor cancellation and revenue KPIs by hotel type and market segment in a recurring management dashboard.",
"6. Improve data governance by standardizing categorical labels and validating dates, numeric ranges, and identifiers at data-entry time.",
"7. Investigate high-value outliers separately instead of deleting them blindly, because extreme bookings may represent genuine premium business."
]
for x in recommendations: print(x)

In [ ]:
print("DATA QUALITY SUMMARY")
print("Original rows:",len(raw))
print("Duplicate rows removed:",raw.duplicated().sum())
print("Rows after cleaning:",len(df))
print("Remaining missing values:",int(df.isna().sum().sum()))
print("Remaining duplicate rows:",df.duplicated().sum())
print("Outlier capping was applied to Lead_Time_Days, ADR, Estimated_Revenue, and Total_Nights using the IQR rule.")